In [ ]:
import jax
import diffrax as dfx
import numpy as np
import jax.numpy as jnp
import plotly.express as px

In [ ]:
def dx_dt(t, x, args):
    A = args['A']
    B = args['B']
    return {
        'a': A @ x['a'],
        'b': B @ x['b']
    }

def sync_term(t, x, args):
    k = args['k']
    i = args['sync'][0]
    j = args['sync'][1]
    e = k*(x['a'][i] - x['b'][j])
    return {
        'a': - 0*e,
        'b': + e
    }


In [ ]:
KEY = jax.random.PRNGKey(42)
def random(*shape):
    global KEY
    KEY, subkey = jax.random.split(KEY)
    return jax.random.normal(subkey, shape)

def symMat(dim):
    A = jnp.array(random(dim, dim) > 0.2, jnp.float32)
    return (A - A.T)


In [ ]:
import diffrax

dim = 10
scale = 0.1
initial = {'a': scale*random(dim), 'b': random(dim)*scale}

Q = symMat(dim)
px.imshow(Q).show()
hab_ind = abs(Q).sum(axis=1).argmax()
print(f"Most connected node: {hab_ind} with {abs(Q).sum(axis=1).max()} connections")
Q = Q / dim
args = {
    'A': Q,
    'B': Q,
    'sync': (hab_ind, hab_ind),
    'k': 3
}

assert abs(initial['a'] - initial['b']).sum() != 0

In [ ]:
term = dfx.MultiTerm(dfx.ODETerm(dx_dt), dfx.ODETerm(sync_term))
solver = dfx.KenCarp3()
t_end = 500
saveat = diffrax.SaveAt(ts=jnp.linspace(0, t_end, 500))
stepsize_controller = diffrax.PIDController(rtol=1e-3, atol=1e-6)
sol = diffrax.diffeqsolve(term, solver, 0.0, t_end, args = args, y0 = initial, dt0 = 0.005, saveat = saveat, max_steps=100_000, stepsize_controller=stepsize_controller)

In [ ]:

def plot_results(sol, hub_ind):
    px.line(sol.ys['a'], x = sol.ts, y = [*range(dim)], labels={'x': 'Time', 'y': 'State a'}, title='State a over time').show()
    px.line(sol.ys['b'], x = sol.ts, y = [*range(dim)], labels={'x': 'Time', 'y': 'State b'}, title='State b over time').show()

    px.line(
        {
            'hubA': sol.ys['a'][:, hub_ind],
            'hubB': sol.ys['b'][:, hub_ind]
        },
        x=sol.ts,
        y = ['hubA', 'hubB'],
        labels={'x': 'Time', 'value': 'State', 'variable': 'Hub'},
        title='hubA vs HubB'
    ).show()

def plot_sync(sol):
    e = ((sol.ys['a'] - sol.ys['b'])**2).mean(axis=1)
    diff = sol.ys['a'] - sol.ys['b']
    px.line(x = sol.ts, y = e, labels={'x': 'Time', 'y': 'Synchronization Error'}, title='Synchronization Error over time').show()
    px.line(diff, x = sol.ts, y = [*range(dim)], labels={'x': 'Time', 'y': 'Synchronization diff'}, title='Synchronization diff over time').show()

In [ ]:
sol.ys['a'].shape

In [ ]:
plot_results(sol, hub_ind=hab_ind)
plot_sync(sol)